In [1]:
import pandas as pd
import joblib

In [22]:
# Load raw (unencoded, unscaled) test set
X_test_raw = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data1/X_test_beforeenc.csv")
y_test_raw = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data1/y_test_beforeenc.csv")

# Load final encoded + scaled test set
X_test_final = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data1/X_test_final.csv")
y_test_final = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data1/y_test_final.csv")

X_train = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data1/X_train_beforeenc.csv")
y_train = pd.read_csv("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/data1/y_train_beforeenc.csv").squeeze()

# Preview shapes to confirm successful loading
print("Raw shapes:", X_test_raw.shape, y_test_raw.shape)
print("Final shapes:", X_test_final.shape, y_test_final.shape)


Raw shapes: (272380, 99) (272380, 1)
Final shapes: (272380, 16) (272380, 1)


### Stress Testing Step 1: Economic Shock Simulation

We simulate a downturn scenario by applying realistic macroeconomic shocks to the test data:

#### Economic Shocks Applied:

- **Income Reduction**  
  `annual_inc *= 0.85`  
  → Simulates a 15% decrease in borrower income, e.g., due to unemployment, pay cuts, or recession impact.

- **DTI Increase**  
  `dti *= 1.25`  
  → Simulates a 25% increase in borrower debt-to-income ratios as income drops or new debt accumulates.

- **Interest Rate Hike**  
  `int_rate *= 1.10`  
  → Simulates a 10% increase in interest rates to reflect a tightening monetary policy environment.

#### Purpose:
These shocks reflect typical stress test conditions imposed by regulators or internal risk teams to assess how the portfolio or model performs under adverse macroeconomic conditions.


In [5]:
import numpy as np

# Create a copy for shock simulation
X_test_shocked = X_test_raw.copy()

# Apply economic shocks
X_test_shocked['annual_inc'] *= 0.85                     # 15% drop in income
X_test_shocked['dti'] *= 1.25                            # 25% increase in DTI
X_test_shocked['int_rate'] = X_test_shocked['int_rate'].astype(str).str.replace('%', '').astype(float)
X_test_shocked['int_rate'] *= 1.10                       # 10% increase in interest rate

print("Stress scenario applied.")


Stress scenario applied.


In [6]:
xgb_model = joblib.load("C:/Users/siddh/AppData/Local/Microsoft/WindowsApps/files/credit-risk-optimization/credit-risk-optimization/models/xgb_credit_model_final.pkl")

In [19]:
cat_cols = X_test_shocked.select_dtypes(include=['object']).columns.tolist()
multi_cat_cols =[col for col in cat_cols if X_test_shocked[col].nunique()>2]
num_cols = [col for col in X_test_shocked.select_dtypes(exclude=['object']).columns.tolist() if col not in ['id', 'loan_condition_int']]


In [16]:
import json
# Load selected features
with open("selected_features.json", "r") as f:
    selected_features = json.load(f)

X_test_shocked_selected = X_test_shocked[selected_features]


In [20]:
from category_encoders import TargetEncoder
from sklearn.preprocessing import StandardScaler

In [23]:
# X_train = X_train[selected_features]

# Fit encoder
encoder = TargetEncoder()
X_train_encoded = X_train.copy()
X_train_encoded[cat_cols] = encoder.fit_transform(X_train[cat_cols], y_train)

# Fit scaler
scaler = StandardScaler()
X_train_encoded[num_cols] = scaler.fit_transform(X_train_encoded[num_cols])

In [25]:
X_test_shocked[cat_cols] = encoder.transform(X_test_shocked[cat_cols])

# 5. Apply scaler (numerical)
X_test_shocked[num_cols] = scaler.transform(X_test_shocked[num_cols])

# 6. Finalize test set with selected features
X_test_shocked_final = X_test_shocked[selected_features]

In [27]:
y_pred_proba_shocked = xgb_model.predict_proba(X_test_shocked_final)[:, 1]

### Stress Test Evaluation (Threshold-Based Strategy) : Bank style rates

This block simulates the impact of a stress scenario by:

- Applying my trained model to shocked data
- Recalculating profit and approval metrics using:
  - LGD = 0.65
  - Threshold = 0.15
  - Risk-based interest rate: `rate = r₀ + α × PD` (with α = 1.5)
- Metrics calculated:
  - Total expected profit under stress
  - Approval rate post-shock
  - Average profit per loan
  - Realized default rate on approved loans


In [49]:
# Assuming LGD and threshold from my best-case model for banking
LGD = 0.65
threshold = 0.15

# Reload EAD (loan amounts) and true labels
loan_amounts = X_test_raw['funded_amnt'].values
y_test = y_test_final.values.ravel()

# Apply threshold to get approved loans
approved_mask_shocked = y_pred_proba_shocked < threshold
approved_pd = y_pred_proba_shocked[approved_mask_shocked]
approved_ead = loan_amounts[approved_mask_shocked]
approved_y = y_test[approved_mask_shocked]

base_rate = 0.07  #base rates increase under stress
alpha = 1.5  # best-case alpha from my simulations

cost_per_loan = 500

interest_rates = base_rate + alpha * approved_pd
expected_revenue = interest_rates * approved_ead
expected_loss = approved_pd * LGD * approved_ead
expected_profit = expected_revenue - expected_loss - cost_per_loan

# Portfolio metrics under shock
portfolio_profit_shocked = expected_profit.sum()
approval_rate_shocked = approved_mask_shocked.mean()
avg_profit_per_loan_shocked = expected_profit.mean()
default_rate_shocked = approved_y.mean()

In [50]:
print("=== Stress Test Results ===")
print(f"Total Profit (Shocked): ${portfolio_profit_shocked:,.2f}")
print(f"Approval Rate (Shocked): {approval_rate_shocked:.2%}")
print(f"Average Profit per Loan (Shocked): ${avg_profit_per_loan_shocked:,.2f}")
print(f"Default Rate on Approved Loans (Shocked): {default_rate_shocked:.2%}")

=== Stress Test Results ===
Total Profit (Shocked): $148,109,787.97
Approval Rate (Shocked): 48.69%
Average Profit per Loan (Shocked): $1,116.69
Default Rate on Approved Loans (Shocked): 1.39%


### Comparison: Baseline vs Stress Test (α = 1.5, Threshold = 0.15)

#### Baseline Results (Normal Economic Conditions)

- **Threshold Used:** 0.15  
- **Alpha (Risk Premium):** 1.5  
- **Base Rate (r₀):** 4%  
- **LGD:** 0.65  
- **Expected Profit:** **$96.7M**  
- **Realized Profit:** **$137.0M**  
- **Average Profit per Loan:** **$660.94**  
- **Approval Rate:** **53.7%**  
- **Default Rate (Approved Loans):** **1.25%**



#### Stress Test Results (Shocked Economic Scenario)

- **Shock Applied To:**  
  - Income ↓  
  - DTI ↑  
  - Interest Rate ↑ (from 4% ➝ **7%** base rate)

- **Threshold Used:** 0.15  
- **Alpha (Risk Premium):** 1.5  
- **Base Rate (r₀):** **7%**  
- **LGD:** 0.65  

- **Total Portfolio Profit:** **$148.1M**  
- **Average Profit per Loan:** **$1,116.69**  
- **Approval Rate:** **48.69%**  
- **Default Rate (Approved Loans):** **1.39%**



#### Comparison vs Baseline

| Metric                   | Baseline       | Stress Test    | Change        |
|-------------------------|----------------|----------------|---------------|
| **Portfolio Profit**    | $137.0M        | $148.1M        | **+ $11.1M**  |
| **Avg Profit per Loan** | $660.94        | $1,116.69      | **+ $455.75** |
| **Approval Rate**       | 53.7%          | 48.7%          | **– 5.0%**    |
| **Default Rate**        | 1.25%          | 1.39%          | **+ 0.14%**   |


#### Insights

- Even under economic stress, **portfolio profit increased**, aided by:
  - Lower approval volume (stricter filtering)
  - **Base rate increase from 4% ➝ 7%**, boosting interest income
- **Per-loan profitability rose sharply**, offsetting the modest increase in default rate
- Shows the effectiveness of **adaptive pricing** under macroeconomic shocks
- Validates the use of **risk-sensitive pricing + tight thresholds** as a stress-resilient lending framework

> This result reflects robust stress-tested pricing mechanics — a key principle in internal capital planning (ICAAP) and regulatory simulations like CCAR.


---

### Stress Test Evaluation (Threshold-Based Strategy) : Competitive rates

This block simulates the impact of a stress scenario by:

- Applying my trained model to shocked data
- Recalculating profit and approval metrics using:
  - LGD = 1
  - Threshold = 0.4
  - Risk-based interest rate: `rate = r₀ + α × PD` (with α = 3)
- Metrics calculated:
  - Total expected profit under stress
  - Approval rate post-shock
  - Average profit per loan
  - Realized default rate on approved loans

In [46]:
import pandas as pd
import numpy as np


LGD = 1
threshold = 0.4
alpha = 3.0
base_rate = 0.06 

# Reload raw test set and labels
loan_amounts = pd.read_csv("X_test_beforeenc.csv")['funded_amnt'].values
y_test = pd.read_csv("y_test_final.csv").values.ravel()

# Approved mask under stress
approved_mask_shocked = y_pred_proba_shocked < threshold
approved_pd = y_pred_proba_shocked[approved_mask_shocked]
approved_ead = loan_amounts[approved_mask_shocked]
approved_y = y_test[approved_mask_shocked]

# Stress-time pricing and profit
max_rate = 0.36  # 36% annual max
interest_rates = np.minimum(base_rate + alpha * approved_pd, max_rate)

cost_per_loan = 500  # or whatever cost you assume


# interest_rates = base_rate + alpha * approved_pd
expected_revenue = interest_rates * approved_ead
expected_loss = approved_pd * LGD * approved_ead
expected_profit = expected_revenue - expected_loss - cost_per_loan

# Portfolio metrics under shock
portfolio_profit_shocked = expected_profit.sum()
approval_rate_shocked = approved_mask_shocked.mean()
avg_profit_per_loan_shocked = expected_profit.mean()
default_rate_shocked = approved_y.mean()

# Display results
print("=== Stress Test Results for Risky Pricing Strategy ===")
print(f"Total Profit (Shocked): ${portfolio_profit_shocked:,.2f}")
print(f"Approval Rate (Shocked): {approval_rate_shocked:.2%}")
print(f"Average Profit per Loan (Shocked): ${avg_profit_per_loan_shocked:,.2f}")
print(f"Default Rate on Approved Loans (Shocked): {default_rate_shocked:.2%}")


=== Stress Test Results for Risky Pricing Strategy ===
Total Profit (Shocked): $271,781,531.58
Approval Rate (Shocked): 65.15%
Average Profit per Loan (Shocked): $1,531.53
Default Rate on Approved Loans (Shocked): 2.73%


### Stress Test Results of Risky Lending (α = 3.0, Threshold = 0.40)

#### Baseline Results (Normal Economic Conditions)

- **Threshold Used:** 0.40  
- **Alpha (Risk Premium):** 3.0  
- **Base Rate (r₀):** 4%  
- **LGD:** 0.65  

- **Portfolio Profit:** **$329.1M**  
- **Average Profit per Loan:** **$1,791.28**  
- **Approval Rate:** **67.46%**  
- **Approved Loans:** **183,739**


#### Stress Test Results (Shocked Economic Scenario)

- **Shock Applied To:**  
  - Income ↓  
  - DTI ↑  
  - Base Rate ↑ from **4% ➝ 7%**

- **Threshold Used:** 0.40  
- **Alpha (Risk Premium):** 3.0  
- **LGD:** 0.65  

- **Portfolio Profit:** **$271.8M**  
- **Average Profit per Loan:** **$1,531.53**  
- **Approval Rate:** **65.15%**  
- **Default Rate (Approved Loans):** **2.73%**


#### Comparison vs Baseline

| Metric                   | Baseline       | Stress Test    | Change         |
|-------------------------|----------------|----------------|----------------|
| **Portfolio Profit**    | $329.1M        | $271.8M        | **– $57.3M**   |
| **Avg Profit per Loan** | $1,791.28      | $1,531.53      | **– $259.75**  |
| **Approval Rate**       | 67.46%         | 65.15%         | **– 2.31%**    |
| **Default Rate**        | (N/A)          | 2.73%          | ↑ (High risk)  |


#### Insights: Why Risky Lending Failed Under Stress

- Despite a **high base interest rate (7%)** and **aggressive pricing (α = 3.0)**, profits **dropped sharply** under stress.
- The model continued approving a large volume of risky loans even as borrower conditions deteriorated.
- **Default rates more than doubled**, eroding profitability.
- This confirms that **risky lending strategies lack resilience** during downturns — they may look lucrative in benign conditions but **collapse under stress**.
- Interest rate hikes **cannot fully compensate** for elevated credit risk in a weakened economy.

> Takeaway: Risk-sensitive pricing is not enough — without tight underwriting thresholds, even aggressive interest rate hikes fail to protect against systemic risk. Prudent approval policies are **crucial to long-term stability**.
